# Phase 7 — Grounded RAG Pipeline (Colab)

Hybrid retrieval + Llama-3.1-8B-Instruct 4-bit grounded generator over
the Phase 3 IKS corpus (285 chunks: 78 Vrikshayurveda + 207 Brihat
Samhita 12-chapter subset).

## Design (master plan §17)

- **Retrieval:** dense (BAAI/bge-large-en-v1.5 over ChromaDB) +
  sparse (BM25) → Reciprocal-Rank-Fusion (k=60) → cross-encoder
  rerank (BAAI/bge-reranker-base). All three stages are toggleable
  for Phase 11 §27 ablations.
- **Generation:** Llama-3.1-8B-Instruct in 4-bit (nf4 + double-quant
  + bf16 compute). Master plan §17 grounded-advisor system prompt:
  answer ONLY from retrieved passages, cite source + chapter + verse,
  step-by-step organic protocol, refuse out-of-corpus questions.
- **Corpus transport:** chunks live in the private HF dataset
  `ankit-iiitdmj/iks-corpus-chunks` (the laptop's `corpus/vector_db/`
  cannot reach Colab). This notebook rebuilds ChromaDB in-session
  from the dataset.

## Platform

Phase 7 runs on Colab (Linux). On Windows, `chromadb` and `torch` /
`sentence_transformers` segfault in the same Python process — that's
captured in the memory entry `feedback-chromadb-torch-windows-dll`.
On Linux the single-process design is fine.

## ⚠️ Before you start

- **Runtime:** GPU (T4 free tier is enough for 4-bit 8B; expect
  ~5–6 GB VRAM).
- **HF Hub token:** Write token belonging to `ankit-iiitdmj` (needed
  for the private chunks dataset AND the gated Llama-3.1 weights).
- **Llama-3.1 license:** you must have accepted it at
  https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct before
  Cell 9 runs. If you haven't, Cell 9 will fail with a 403 on the
  weights download.


In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL = "https://github.com/ankit8453/iks-rag-thesis.git"
REPO_PATH = "/content/iks-rag-thesis"

import os, subprocess, sys

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "requirements.txt")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
print("Repo root contents:", sorted(os.listdir(".")))

# Phase 7 runtime deps. Colab pre-installs torch / numpy / pandas /
# sklearn so we skip those. `bitsandbytes` is needed for 4-bit
# Llama; `rank-bm25` for sparse retrieval.
_pip_packages = [
    "transformers>=4.44",
    "bitsandbytes>=0.43",
    "accelerate>=0.33",
    "sentence-transformers>=3.0",
    "chromadb>=0.5",
    "rank-bm25>=0.2.2",
    "huggingface_hub>=0.24",
    "datasets>=2.20",
    "pyyaml>=6.0",
]
proc = subprocess.run(
    [sys.executable, "-m", "pip", "install", *_pip_packages],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print("PIP STDOUT (tail):\n" + proc.stdout[-3000:])
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tails above.")
print("setup ok")


Repo root contents: ['.git', '.gitattributes', '.gitignore', 'BACKUP.md', 'HF_UPLOAD_REPORT.md', 'README.md', 'configs', 'corpus', 'data', 'demo', 'environment.yml', 'literature_tracker.csv', 'models', 'notebooks', 'notes', 'paper', 'progress.md', 'pytest.ini', 'references.bib', 'requirements.txt', 'research_journal', 'results', 'scripts', 'src', 'tests', 'thesis']
setup ok


In [ ]:
# Cell 3 — HF Hub login (private chunks dataset + gated Llama weights)
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
print(f"HF Hub ok: user={_whoami['name']}")


HF Hub ok: user=ankit-iiitdmj


In [ ]:
# Cell 4 — GPU check (4-bit 8B Llama needs ~5–6 GB; T4 is fine)
import subprocess, torch, sys
sys.path.insert(0, REPO_PATH)

subprocess.run(["nvidia-smi"], check=False)
print()
assert torch.cuda.is_available(), "No GPU detected — switch runtime to GPU before running Cell 9."
dev = torch.cuda.get_device_properties(0)
vram_gib = dev.total_memory / 1024**3
print(f"GPU: {dev.name}, VRAM: {vram_gib:.1f} GiB")
if vram_gib < 14:
    print("WARN: <14 GiB VRAM — Llama-3.1-8B 4-bit may OOM on contexts; consider "
          "`model_name=meta-llama/Llama-3.2-3B-Instruct` when constructing the generator.")



GPU: Tesla T4, VRAM: 14.6 GiB


## Corpus rebuild

Pull the private 285-row chunks dataset from HF Hub and re-embed it
into a fresh ChromaDB collection at `corpus/vector_db/`. Re-running
the cell upserts by deterministic sha1 `chunk_id` — no duplicates.


In [ ]:
# Cell 6 — Load chunks from HF and rebuild ChromaDB locally
from src.rag.corpus_loader import load_chunks_from_hf, build_chroma

chunks = load_chunks_from_hf()  # ankit-iiitdmj/iks-corpus-chunks
print(f"Loaded {len(chunks)} chunks; first chunk:")
print(" ", {k: chunks[0][k] for k in ("book_id", "chapter", "verse_or_section", "chunk_id")})

collection = build_chroma(chunks, persist_dir="corpus/vector_db")
print(f"\nCollection populated: count={collection.count()}")
assert collection.count() == len(chunks), "Chroma count mismatch — re-run Cell 6."


INFO:numexpr.utils:NumExpr defaulting to 2 threads.
2026-06-04T17:55:50 | INFO    | numexpr.utils | NumExpr defaulting to 2 threads.
INFO:datasets:TensorFlow version 2.20.0 available.
2026-06-04T17:55:51 | INFO    | datasets | TensorFlow version 2.20.0 available.
INFO:datasets:JAX version 0.7.2 available.
2026-06-04T17:55:51 | INFO    | datasets | JAX version 0.7.2 available.
INFO:src.rag.corpus_loader:Loading chunks from ankit-iiitdmj/iks-corpus-chunks (split=train) ...
2026-06-04T17:55:52 | INFO    | src.rag.corpus_loader | Loading chunks from ankit-iiitdmj/iks-corpus-chunks (split=train) ...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/ankit-iiitdmj/iks-corpus-chunks/resolve/main/README.md "HTTP/1.1 200 OK"
2026-06-04T17:55:52 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/datasets/ankit-iiitdmj/iks-corpus-chunks/resolve/main/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/datasets/ankit-iiitdmj/iks-corpus-chunks/resolv

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/ankit-iiitdmj/iks-corpus-chunks/resolve/5529fd24940ada9c9fc8599f336137dffcb642dd/iks-corpus-chunks.py "HTTP/1.1 404 Not Found"
2026-06-04T17:55:52 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/datasets/ankit-iiitdmj/iks-corpus-chunks/resolve/5529fd24940ada9c9fc8599f336137dffcb642dd/iks-corpus-chunks.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ankit-iiitdmj/iks-corpus-chunks/ankit-iiitdmj/iks-corpus-chunks.py "HTTP/1.1 404 Not Found"
2026-06-04T17:55:52 | INFO    | httpx | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ankit-iiitdmj/iks-corpus-chunks/ankit-iiitdmj/iks-corpus-chunks.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/ankit-iiitdmj/iks-corpus-chunks/revision/5529fd24940ada9c9fc8599f336137dffcb642dd "HTTP/1.1 200 OK"
2026-06-04T17:55:52 |

data/train-00000-of-00001.parquet:   0%|          | 0.00/517k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5 "HTTP/1.1 200 OK"
2026-06-04T17:56:31 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-large-en-v1.5 "HTTP/1.1 200 OK"
INFO:src.rag.corpus_loader:Embedding 285 texts in batches of 32 ...
2026-06-04T17:56:32 | INFO    | src.rag.corpus_loader | Embedding 285 texts in batches of 32 ...


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

INFO:src.rag.corpus_loader:Upserting 285 chunks into Chroma collection 'iks_corpus' ...
2026-06-04T17:56:57 | INFO    | src.rag.corpus_loader | Upserting 285 chunks into Chroma collection 'iks_corpus' ...
INFO:src.rag.corpus_loader:  ... upserted 32/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 32/285
INFO:src.rag.corpus_loader:  ... upserted 64/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 64/285
INFO:src.rag.corpus_loader:  ... upserted 96/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 96/285
INFO:src.rag.corpus_loader:  ... upserted 128/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 128/285
INFO:src.rag.corpus_loader:  ... upserted 160/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 160/285
INFO:src.rag.corpus_loader:  ... upserted 192/285
2026-06-04T17:56:58 | INFO    | src.rag.corpus_loader |   ... upserted 192/285
INFO:src.rag.corpus_loader:


Collection populated: count=285


In [ ]:
# Cell 7 — Build HybridRetriever (dense + sparse + reranker, all on)
from src.rag.retriever import HybridRetriever

retriever = HybridRetriever(collection, use_dense=True, use_sparse=True, use_reranker=True)
print(
    f"HybridRetriever ready: dense={retriever.use_dense} "
    f"sparse={retriever.use_sparse} reranker={retriever.use_reranker}"
)


INFO:src.rag.retriever:Loading embedding model BAAI/bge-large-en-v1.5 on cuda ...
2026-06-04T17:57:39 | INFO    | src.rag.retriever | Loading embedding model BAAI/bge-large-en-v1.5 on cuda ...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-06-04T17:57:39 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
2026-06-04T17:57:39 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-large-en-v1.5/d4aa6901d3a41ba39fb536a557fa166f842b0e09/modules.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-04T17:57:41 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-04T17:57:41 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-large-en-v1.5/resolve/main/vi

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-06-04T17:57:44 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-04T17:57:44 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
2026-06-04T17:57:44 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD h

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-06-04T17:57:53 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-06-04T17:57:53 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/special_tokens_map.json "HTTP/1.1 200 OK"
2026-06-04T17:57:53 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-reranker-base/2cfc18c9415c912f9d8155881c133215df768a70/special_tokens_map.json "HTTP

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-06-04T17:57:53 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-reranker-base/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
2026-06-04T17:57:56 | INFO    | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-reranker-base "HTTP/1.1 200 OK"
INFO:src.rag.retriever:HybridRetriever: dense=True sparse=True rerank=True top_k_dense=20 top_k_sparse=20 top_k_rerank=5
2026-06-04T17:57:57 | INFO    | src.rag.retriever | HybridRetriever: dense=True sparse=True rerank=True top_k_dense=20 top_k_sparse=20 top_k_rerank=5


HybridRetriever ready: dense=True sparse=True reranker=True


In [ ]:
# Cell 8 — Retriever smoke. Runs BEFORE Cell 9 so a retrieval failure
# fails fast (no need to pay the ~5 GB Llama download cost).
SMOKE_QUERIES = [
    "how to treat a diseased tree",                    # Vrikshayurveda / Brihat ch.55
    "signs that predict rainfall",                     # Brihat ch.21-28
    "how to find underground water",                   # Brihat ch.54
    "yellow leaf disease and the correct soil for it", # joint disease + soil (Phase 8 preview)
    "organic protocol for sandy loam crops",          # cross-source retrieval
]

for q in SMOKE_QUERIES:
    hits = retriever.retrieve(q, k=5)
    print("=" * 78)
    print(f"QUERY: {q!r}")
    for i, h in enumerate(hits, 1):
        meta = h.metadata or {}
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n', ' ')[:140]
        print(f"  [{i}] score={h.score:.4f} stage={h.retriever}  {src}")
        print(f"        {snip}")
    print()


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: 'how to treat a diseased tree'
  [1] score=0.9272 stage=reranker  Brihat Samhita ch.55 v.section_1
        TAAASTEATA: WM MN Chapter LV—Treatment of Trees  This too is an important scientific topic that was cultivated in ancient India. Gardening i
  [2] score=0.4554 stage=reranker  Vrikshayurveda ch.full v.160.2
        Is7. The diseases of the dafr type can be overcome with mustard, ramallur, cidanga, vice, sina, and water mixed bitter, strang, and astringe
  [3] score=0.3945 stage=reranker  Brihat Samhita ch.55 v.section_4
        horse-gram, black gram, green gram, sesamum and barley. Being treated thus, it will have abundant flowers and fruits. Uianmrmpegqueaies ¢ fa
  [4] score=0.3920 stage=reranker  Brihat Samhita ch.55 v.section_3
        The transplanted (grafted) trees should be watered both in the morning and evening everyday in summer; on alternate days in the cold season;
  [5] score=0.3776 stage=reranker  Vrikshayurveda ch.full v.160.4
        are amnicinsd wl roid 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: 'signs that predict rainfall'
  [1] score=0.8876 stage=reranker  Brihat Samhita ch.28 v.section_1
        ANATMEA: Usk Chapter XXVIII—Signs of Immediate Rain agisaa afaafaad uafsratistca aezt aid atat wafa afe at ert: Yratat | ateag B: WACAEH WTZ
  [2] score=0.8197 stage=reranker  Brihat Samhita ch.28 v.section_2
        Signs of Immediate Rain XXVIII 275 ata zeq equa ate at anit acasae aT aaah aafa afe at atresia at Tet area: afaaafacrater faa qearara aferet
  [3] score=0.7912 stage=reranker  Brihat Samhita ch.28 v.section_3
        aeafra fata gergeata sau quate | THs: TYASA FHT WeeA: Tactics fafada ven If the domestic animals like cows are unwilling to go out of  Signs
  [4] score=0.7739 stage=reranker  Brihat Samhita ch.23 v.section_2
        generally be rain once again under the same stars in the season. If  Rainfall XXIII 247 there be no rain at all in any of the asterisms begi
  [5] score=0.7706 stage=reranker  Brihat Samhita ch.21 v.section_8
        Though modern scien

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: 'how to find underground water'
  [1] score=0.6975 stage=reranker  Brihat Samhita ch.54 v.section_1
        SHTTAT Wy Chapter LIV—Exploration of Water Springs [Like the previous chapter, Water-Divination is an important and  independent science pra
  [2] score=0.0777 stage=reranker  Brihat Samhita ch.54 v.section_18
        [As I have mentioned in the beginning of this chapter Manu and Sarasvata had both written on Dakdrgala. Varadhamihira has summarized the for
  [3] score=0.0485 stage=reranker  Brihat Samhita ch.54 v.section_7
        geg g vata aHet ater gedtlaat aa: wat 1 TE CAATAST: TATA SAAT ATA UIQ If a frog is seen at the foot of any tree, there will be water at a de
  [4] score=0.0452 stage=reranker  Brihat Samhita ch.54 v.section_16
        There will be abundant water at a depth of 20 cubits to the south of trees that are very glossy. The same result should be declared if a tre
  [5] score=0.0388 stage=reranker  Brihat Samhita ch.54 v.section_11
        Exploration of

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: 'yellow leaf disease and the correct soil for it'
  [1] score=0.2561 stage=reranker  Vrikshayurveda ch.full v.160.1
        Ifthe farm is infested with rats, ctc. the following trunk, appearance of knots on the trunk or leaves, and mantra writtenon the plantain le
  [2] score=0.0314 stage=reranker  Brihat Samhita ch.55 v.section_1
        TAAASTEATA: WM MN Chapter LV—Treatment of Trees  This too is an important scientific topic that was cultivated in ancient India. Gardening i
  [3] score=0.0232 stage=reranker  Vrikshayurveda ch.full v.250-231.11
        JA land which is fall of may grass or of keer and apie man-height ane after dipping a distance of one toaea prs and where thw anil is crreMr
  [4] score=0.0169 stage=reranker  Vrikshayurveda ch.full v.154-55
        154. The cotton plant when sprinkled with water mixed otherwise the salt will use his tail, powerful like the |  with fish flesh, ynthi when
  [5] score=0.0165 stage=reranker  Vrikshayurveda ch.full v.section_15
    

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

QUERY: 'organic protocol for sandy loam crops'
  [1] score=0.0099 stage=reranker  Vrikshayurveda ch.full v.280.3
        and planting materials to promote ¢ormination, seedling recommendations, hoping thatthe readers would follow growth, and sevdling care (vers
  [2] score=0.0035 stage=reranker  Vrikshayurveda ch.full v.section_9
        (sod safe acme sar 6A  a Wize  oe | _~ eenmMaTaiMaAT eR IAS VSIA AAA |) | |) AcaqaneecgAAemenmaAGA oy suaahsaasng eran “1 ) HSiaeneaimna ea 
  [3] score=0.0032 stage=reranker  Vrikshayurveda ch.full v.280.8
        pest management melhods—cultural, onganc-chemical, Modern agriculture and | and smoke— were probably used extensively, Vriksh ayurve da The 
  [4] score=0.0026 stage=reranker  Vrikshayurveda ch.full v.2.7
        The Asian Agri-History Foundation (A AHF), a non-profit trust, was established and registered in 1994 to facilitate dissemination of informa
  [5] score=0.0024 stage=reranker  Vrikshayurveda ch.full v.section_21
        with milk an

In [ ]:
# Cell 8.5 — Free retriever VRAM before loading Llama (T4-fit trick).
# Moves the BGE embedder + cross-encoder reranker to CPU so the
# transient bf16 weight-materialization buffer for 8B in 4-bit can fit.
import os, gc, torch

# Allocator fragmentation mitigation. MUST be set before any CUDA allocation
# in the LLM load — runtime restart picks this up cleanest.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

before = torch.cuda.memory_allocated() / 1024**3
if getattr(retriever, "_embedder", None) is not None:
    retriever._embedder.to("cpu")
if getattr(retriever, "_reranker", None) is not None:
    # CrossEncoder wraps the actual transformer in .model
    retriever._reranker.model.to("cpu")
gc.collect()
torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1024**3
print(f"Retriever moved to CPU. VRAM: {before:.2f} → {after:.2f} GiB freed {before-after:.2f} GiB")


Retriever moved to CPU. VRAM: 8.18 → 0.01 GiB freed 8.17 GiB


In [ ]:
# Cell 9 — Load Llama-3.1-8B 4-bit (gated — Cell 3's token must have license access)
import torch
from src.rag.generator import GroundedGenerator

generator = GroundedGenerator(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    load_in_4bit=True,
    temperature=0.2,
    max_new_tokens=512,
    seed=42,
)
generator._ensure_loaded()  # noqa: SLF001 — warm up here so VRAM is visible BEFORE Cell 10
torch.cuda.empty_cache()
mem = torch.cuda.memory_allocated() / 1024**3
print(f"Llama-3.1-8B 4-bit loaded. CUDA memory in use: {mem:.2f} GiB")


INFO:src.rag.generator:Loading tokenizer for meta-llama/Llama-3.1-8B-Instruct ...
2026-06-04T18:32:24 | INFO    | src.rag.generator | Loading tokenizer for meta-llama/Llama-3.1-8B-Instruct ...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-04T18:32:24 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-04T18:32:24 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-04T18:32:24 | INFO    | httpx | HTTP Request: HEAD https://hugging

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-06-04T18:33:33 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-06-04T18:33:33 | INFO    | httpx | HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Llama-3.1-8B 4-bit loaded. CUDA memory in use: 5.32 GiB


In [ ]:
# Cell 9.5 — Move retriever models back to GPU now that Llama steady-state is on VRAM.
import torch
if getattr(retriever, "_embedder", None) is not None:
    retriever._embedder.to("cuda")
if getattr(retriever, "_reranker", None) is not None:
    retriever._reranker.model.to("cuda")
torch.cuda.empty_cache()
print(f"Retriever back on GPU. Total VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")


Retriever back on GPU. Total VRAM: 7.61 GiB


In [ ]:
# Cell 9.7 — Free 8B VRAM and swap to Llama-3.2-3B-Instruct for A/B comparison
import gc, torch

# Drop the 8B generator (and its pipeline)
del generator
try:
    del pipeline
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
print(f"After 8B unload: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")

# Park retriever models on CPU during 3B load (same trick as Cell 8.5)
if getattr(retriever, "_embedder", None) is not None:
    retriever._embedder.to("cpu")
if getattr(retriever, "_reranker", None) is not None:
    retriever._reranker.model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

from src.rag.generator import GroundedGenerator
generator = GroundedGenerator(
    model_name="meta-llama/Llama-3.2-3B-Instruct",   # ← 3B Instruct
    load_in_4bit=True,
    temperature=0.2,
    max_new_tokens=512,
    seed=42,
)
generator._ensure_loaded()

# Move retriever back to GPU now that 3B is steady-state
if getattr(retriever, "_embedder", None) is not None:
    retriever._embedder.to("cuda")
if getattr(retriever, "_reranker", None) is not None:
    retriever._reranker.model.to("cuda")
torch.cuda.empty_cache()
mem = torch.cuda.memory_allocated() / 1024**3
print(f"Llama-3.2-3B-Instruct 4-bit loaded. VRAM in use: {mem:.2f} GiB")

# Rebuild the pipeline with the new generator
from src.rag.pipeline import RAGPipeline
pipeline = RAGPipeline(retriever=retriever, generator=generator, default_k=5)


After 8B unload: 2.29 GiB


INFO:src.rag.generator:Loading tokenizer for meta-llama/Llama-3.2-3B-Instruct ...
2026-06-04T18:53:17 | INFO    | src.rag.generator | Loading tokenizer for meta-llama/Llama-3.2-3B-Instruct ...
INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-04T18:53:17 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"
2026-06-04T18:53:17 | INFO    | httpx | HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-04T18:53:17 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-06-04T18:53:17 | INFO    | httpx | HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-06-04T18:58:50 | INFO    | httpx | HTTP Request: HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"
2026-06-04T18:58:50 | INFO    | httpx | HTTP Request: GET https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/generation_config.json "HTTP/1.1 200 OK"


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

INFO:src.rag.pipeline:RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5
2026-06-04T18:58:51 | INFO    | src.rag.pipeline | RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5


Llama-3.2-3B-Instruct 4-bit loaded. VRAM in use: 4.40 GiB


In [ ]:
# Cell 10 — End-to-end RAG: retriever → grounded generator, 5 demo queries
from src.rag.pipeline import RAGPipeline

pipeline = RAGPipeline(retriever=retriever, generator=generator, default_k=5)

DEMO_QUERIES = [
    # In-corpus expected to work well:
    "How should a diseased tree with falling branches be treated?",
    "What signs in the sky predict imminent rainfall?",
    "How does the classical text guide finding underground water?",
    # Joint disease + soil — foreshadows Phase 8 (multimodal context):
    "What organic protocol should be used for a tree showing yellow leaves growing in mixed sandy-loam soil?",
    # OUT-OF-CORPUS faithfulness check: the model MUST refuse rather than hallucinate.
    "What is the recommended drone-spraying schedule for monoculture rice fields?",
]

for q in DEMO_QUERIES:
    print("=" * 78)
    print(f"QUERY: {q}")
    result = pipeline.answer(q, k=5)
    print("--- ANSWER ---")
    print(result.answer)
    print("--- CITATIONS ---")
    for c in result.citations:
        print("  -", c)
    print("--- CHUNKS USED ---")
    for cid in result.used_chunk_ids:
        # show the metadata of every chunk the model actually cited
        match = next((r for r in result.retrieved if r.chunk_id == cid), None)
        if match is not None:
            meta = match.metadata
            print(f"  {cid[:10]}  {meta.get('source_text')} ch.{meta.get('chapter')} v.{meta.get('verse_or_section')}")
    print("--- TOP RETRIEVED (for inspection) ---")
    for i, h in enumerate(result.retrieved[:3], 1):
        meta = h.metadata
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n',' ')[:120]
        print(f"  [{i}] score={h.score:.4f} {src} :: {snip}")
    print()


INFO:src.rag.pipeline:RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5
2026-06-04T18:35:33 | INFO    | src.rag.pipeline | RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5


QUERY: How should a diseased tree with falling branches be treated?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


--- ANSWER ---
To treat a diseased tree with falling branches, follow these steps:

1. Clear the tree of ulcers and the like with a knife [Vrikshayurveda, ch.full, v.160.3].
2. Apply a paste made of Vidanga, ghee, and silt to the affected parts [Vrikshayurveda, ch.full, v.160.3].
3. Sprinkle the affected parts with water and milk [Vrikshayurveda, ch.full, v.160.3].
4. If the branches are burnt, cut them off quickly and smoke the tree with the mixture of white mustard, milk, and water [Vrikshayurveda, ch.full, v.160.3].
5. Sprinkle the particular spots with water and milk [Vrikshayurveda, ch.full, v.160.3].
6. Smoke the tree with the shells of the crab [Vrikshayurveda, ch.full, v.160.3].
7. Anoint the tree with a mixture of sugar, sesame, and milk, each measuring one flake (256 handiuls), barley, and sesame, fused for watering and anointing [Vrikshayurveda, ch.full, v.207].
8. Water the tree with milk that has been cooled after being boiled, with the fruits of a tree destroyed (prematur

...

--- ANSWER ---
To find underground water, follow these steps:

1. Look for places with Muifija grass, reeds, and sacrificial grass, where the earth is blue and mixed with pebbles, or where the earth is black or red [Brihat Samhita, ch.54, v.section_19].
2. Check if the earth is copper-coloured mixed with gravel, which will yield water of astringent taste; red-brown earth makes the water brackish; a pale white earth produces salt-water; and blue earth, sweet water [Brihat Samhita, ch.54, v.section_19].
3. Identify trees such as Saka, Asvakarna, Arjuna, Bilva, Sarja, Sriparni, Arista, Dhava, Sixhsapa, and others that are coarse and have leaves full of holes, which indicate the existence of water far away [Brihat Samhita, ch.54, v.section_19].
4. Observe the land which has the colour of the Sun, fire, ashes, camel, or donkey, is said to be waterless [Brihat Samhita, ch.54, v.section_19].
5. Check if there are trees, bushes, and creepers that are glossy and densely covered with leaves, or 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ANSWER ---
Based on the retrieved passages, the protocol for treating a tree showing yellow leaves growing in mixed sandy-loam soil is as follows:

1. **Diagnose the cause of yellowing leaves**: According to [Vrikshayurveda, ch.full, v.160.1], yellowing of leaves can be caused by excessive watering with materials that are sweet, oily, sour, or cold, or by the imbalance of the elements (pitta, vata, and kapha) [Vrikshayurveda, ch.full, v.160.1].
2. **Check for other symptoms**: Observe the tree for other symptoms such as paleness of leaves, untimely dropping of fruits, dryness, and lack of original fragrance [Vrikshayurveda, ch.full, v.160.1].
3. **Remove excessive water**: If the tree is excessively watered, reduce watering to prevent further damage [Vrikshayurveda, ch.full, v.280.10].
4. **Apply a treatment**: For diseases caused by an imbalance of pitta, apply a treatment by sprinkling water mixed with ash, ghee, and honey on the roots [Vrikshayurveda, ch.full, v.186].
5. **Use a

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ANSWER ---
The retrieved classical-text passages do not contain enough information to answer this question. Please consult a qualified agricultural expert.
--- CITATIONS ---
--- CHUNKS USED ---
--- TOP RETRIEVED (for inspection) ---
  [1] score=0.0002 Brihat Samhita ch.55 v.section_4 :: horse-gram, black gram, green gram, sesamum and barley. Being treated thus, it will have abundant flowers and fruits. Ui
  [2] score=0.0002 Vrikshayurveda ch.full v.2.2 :: birbhatd (Cirblatr cucu) 155 dinar (chen) (Anogeisces Letifotie Wall, ex Bedtd,, ghatti, | black gram (Veg menage (Lien)
  [3] score=0.0001 Vrikshayurveda ch.full v.2.5 :: paravata (Cicéa acide (Linn) Morr, country gooseberry) 95, nitbhe (Wise paradisicey Linn. banana, plantain) 20 (alsosre 



In [ ]:
# Cell 10 — End-to-end RAG: retriever → grounded generator, 5 demo queries
from src.rag.pipeline import RAGPipeline

pipeline = RAGPipeline(retriever=retriever, generator=generator, default_k=5)

DEMO_QUERIES = [
    # In-corpus expected to work well:
    "How should a diseased tree with falling branches be treated?",
    "What signs in the sky predict imminent rainfall?",
    "How does the classical text guide finding underground water?",
    # Joint disease + soil — foreshadows Phase 8 (multimodal context):
    "What organic protocol should be used for a tree showing yellow leaves growing in mixed sandy-loam soil?",
    # OUT-OF-CORPUS faithfulness check: the model MUST refuse rather than hallucinate.
    "What is the recommended drone-spraying schedule for monoculture rice fields?",
]

for q in DEMO_QUERIES:
    print("=" * 78)
    print(f"QUERY: {q}")
    result = pipeline.answer(q, k=5)
    print("--- ANSWER ---")
    print(result.answer)
    print("--- CITATIONS ---")
    for c in result.citations:
        print("  -", c)
    print("--- CHUNKS USED ---")
    for cid in result.used_chunk_ids:
        # show the metadata of every chunk the model actually cited
        match = next((r for r in result.retrieved if r.chunk_id == cid), None)
        if match is not None:
            meta = match.metadata
            print(f"  {cid[:10]}  {meta.get('source_text')} ch.{meta.get('chapter')} v.{meta.get('verse_or_section')}")
    print("--- TOP RETRIEVED (for inspection) ---")
    for i, h in enumerate(result.retrieved[:3], 1):
        meta = h.metadata
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n',' ')[:120]
        print(f"  [{i}] score={h.score:.4f} {src} :: {snip}")
    print()


INFO:src.rag.pipeline:RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5
2026-06-04T18:59:00 | INFO    | src.rag.pipeline | RAGPipeline ready: retriever=HybridRetriever generator=GroundedGenerator default_k=5


QUERY: How should a diseased tree with falling branches be treated?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ANSWER ---
To treat a diseased tree with falling branches, follow these steps:

1. Clear the tree of ulcers and the like (i.e., whatever is colourless and wet) with a knife. [Vrikshayurveda, ch.full, v.280.11]
2. Apply a paste made of Vidanga, ghee, and silt to the affected parts. [Vrikshayurveda, ch.full, v.280.11]
3. Sprinkle water and milk on the affected parts. [Vrikshayurveda, ch.full, v.280.11]
4. If the branches are burnt, cut them off quickly by smoking the tree with the mixture of white and the particular spots should be sprinkled with water and milk and smoked with the shells of the crab. [Vrikshayurveda, ch.full, v.195]
5. If the tree is scorched with fire, water it with a mixture of sugar, sesame, and milk, each measuring one flake (256 handfuls), barley, and fused for watering and anointing. [Vrikshayurveda, ch.full, v.207]
6. Anoint the tree with the mixture of sesame, barley, and milk, and water, and strike it with lightning with cold mixture (7) of sesame, barley, a

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

--- ANSWER ---
To find underground water, follow these steps:

1. Look for trees with glossy leaves and extensive canopies, as they indicate the presence of water nearby (Vrikshayurveda, ch.full, v.250-231.9). These trees include Tilaka (Tila), Amrataka (Spondias), Varuna (Tapia), Bhallataka (Marking Nut tree), Bilva, Tinduka (a sort of ebony), Okola (walnut), Pindara, Sirisa Siris), Afijana, Parisaka, Vaiijula (Bayas), and Atibala (Source 5).

2. Check for ant-hills on these trees, as they indicate the presence of water at a depth of 22 cubits and 3 cubits to the north of the tree (Source 5).

3. If you find a waterless region with canes growing in it, construct a well and dig to a depth of half a man-height (Source 5).

4. Alternatively, if you find a pond with a depth of half a man-height and a frog of whitish color and yellowish soil, it may indicate the presence of water (Source 5).

5. If you find a pond with a depth of half a man-height and a reflection of the garden in its extr

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ANSWER ---
To address the user's question regarding the organic protocol for a tree showing yellow leaves growing in mixed sandy-loam soil, we need to analyze the retrieved passages.

First, we need to identify the possible causes of yellow leaves in trees, as mentioned in the passages. According to [Source 2, Vrikshayurveda, ch.160.1, v.179], diseases such as yellowing of leaves, premature fruiting, and leaflessness can occur due to an imbalance of the three doshas (pitta, vata, and kapha) in the body. Additionally, [Source 2, Vrikshayurveda, ch.160.1, v.181] mentions that yellowing of leaves can also be caused by the attack of ants, indigestion, and dryness.

To address the issue, we need to consider the treatment options mentioned in the passages. According to [Source 2, Vrikshayurveda, ch.160.1, v.181], the use of bitter substances, such as ghee, and the application of white mustard paste can help reduce scorching heat or frost. However, these treatments may not be directly app

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

--- ANSWER ---
The retrieved classical-text passages do not contain enough information to answer this question. Please consult a qualified agricultural expert.
--- CITATIONS ---
--- CHUNKS USED ---
--- TOP RETRIEVED (for inspection) ---
  [1] score=0.0002 Brihat Samhita ch.55 v.section_4 :: horse-gram, black gram, green gram, sesamum and barley. Being treated thus, it will have abundant flowers and fruits. Ui
  [2] score=0.0002 Vrikshayurveda ch.full v.2.2 :: birbhatd (Cirblatr cucu) 155 dinar (chen) (Anogeisces Letifotie Wall, ex Bedtd,, ghatti, | black gram (Veg menage (Lien)
  [3] score=0.0001 Vrikshayurveda ch.full v.2.5 :: paravata (Cicéa acide (Linn) Morr, country gooseberry) 95, nitbhe (Wise paradisicey Linn. banana, plantain) 20 (alsosre 



## How to read these results

- **In-corpus queries 1–3** should each produce a numbered organic
  protocol with one or more `[Source Text, ch.X, v.Y]` citations
  matching the top retrieved chunks. The `CHUNKS USED` row tells you
  which retrieved chunks the answer actually cited.
- **Joint query 4** is a preview of Phase 8 (multimodal context):
  the question deliberately mentions both a disease symptom and a
  soil type, so retrieval should pull chunks from both Vrikshayurveda
  (disease) and Brihat Samhita (soil/exploration). The Phase 8
  notebook will inject vision-module predictions into the query at
  exactly this seam.
- **Out-of-corpus query 5** is the §17 faithfulness sanity check.
  The model MUST emit the locked refusal sentence — *"The retrieved
  classical-text passages do not contain enough information to
  answer this question. Please consult a qualified agricultural
  expert."* — rather than hallucinating a drone-spraying schedule.
  If it hallucinates a treatment instead, log it as a Phase 11 RAGAS
  faithfulness issue and do NOT hide it.


## Phase 7 complete

Pipeline lives at:

- `src/rag/corpus_loader.py` (`load_chunks_from_hf`, `build_chroma`)
- `src/rag/retriever.py` (`HybridRetriever`, `RetrievedChunk`)
- `src/rag/generator.py` (`GroundedGenerator`, §17 prompt)
- `src/rag/pipeline.py` (`RAGPipeline`)

### Next: Phase 8 (multimodal integration)

The query-construction step is the seam — Phase 8 will compose the
user question with the Phase 5 disease classifier's prediction and
the Phase 6 soil classifier's prediction (e.g. *"yellow leaves on
Bottle Gourd in sandy-loam soil"*) and route the enriched query
through this same `RAGPipeline`.

### Swapping LLMs

If VRAM is tight (T4 free tier with long contexts), pass
`model_name="meta-llama/Llama-3.2-3B-Instruct"` to either
`GroundedGenerator(...)` or `RAGPipeline(model_name=...)`. The pipeline
is generator-agnostic, so the rest of the code is unchanged.

### Adding more books

Once Krishi Parashara, Upavanavinoda, Kashyapiyakrishisukti, or the
TBD-sixth text are processed by Phase 3, re-run
`python scripts/push_corpus_chunks.py` on the laptop to push the
expanded corpus to `ankit-iiitdmj/iks-corpus-chunks`. The next Colab
Cell 6 run picks them up automatically — no code change anywhere in
the RAG pipeline.


---

# Phase 3b.2 re-run -- Gemini-OCR'd corpus (comparison vs above)

> All cells above this point were produced from the **Tesseract**-OCR'd
> 285-chunk corpus (Phase 3, May 2026). The outputs you see frozen above
> in cells 6-10 are from that run. They are kept here intentionally so
> the comparison is visible in one document.

**What changed in Phase 3b.2 (June 2026):**

1. **New OCR engine** -- the four book PDFs were re-OCR'd with
   **Gemini 3.5 Flash** (paid tier, ~Rs 41 total spend) instead of
   local Tesseract. Gemini was prompted to skip Devanagari script,
   preserve verse numbers exactly, and use context to spell Sanskrit
   plant names consistently. The Tesseract output had repeating
   Devanagari ink-bleed noise inside English sentences which was
   degrading retrieval and refusal behaviour.
2. **Two new books added.** The corpus now spans **four** classical
   treatises, not two:
   - Vrikshayurveda (Surapala, tr. Sadhale 1996)
   - Brihat Samhita Part 1 (Varahamihira, tr. Bhat, 12 chapters)
   - **NEW**: Krishi Parashara (Majumdar & Banerji 1960, PDF pp.94-119)
   - **NEW**: Upavanavinoda (Sarngadhara, tr. Majumdar, PDF pp.77-96)
3. **Fewer but cleaner chunks**: 285 -> 206. Tesseract's mis-OCR'd
   numbers were producing spurious "new verse" boundaries; Gemini's
   clean output lets the chunker pack proper semantic blocks.

**How to read the cells below:**

The next 4 code cells rebuild the pipeline with the new corpus and
re-run **the exact same 5 demo queries** that appear in cell 10
above. Compare answers, citations, and retrieved-chunk snippets
side-by-side with the Tesseract-era outputs above.

> **Note on running this section:** these cells are self-contained --
> they re-import everything and rebuild a fresh retriever
> (`retriever_gemini`) and pipeline (`pipeline_gemini`) at a separate
> ChromaDB path (`corpus/vector_db_gemini/`). If you want to preserve
> the frozen outputs in cells 6-10, do **not** re-run those cells in
> Colab -- start execution from this section by clicking on the next
> cell and using `Runtime -> Run from selected cell`. You will need
> to re-run **only** cell 3 (HF login) once before this section so
> the auth token is in memory.


In [ ]:
# Phase 3b.2 -- Cell A: force-reload chunks from HF (the dataset was
# replaced after the Gemini re-OCR, so the on-disk HF cache must be
# bypassed). Expect 206 chunks across 4 books.
from datasets import load_dataset
from src.rag.corpus_loader import DEFAULT_CHUNKS_REPO, REQUIRED_FIELDS

ds_gemini = load_dataset(
    DEFAULT_CHUNKS_REPO,
    split="train",
    download_mode="force_redownload",   # bypass any old-snapshot cache
)
chunks_gemini = []
for row in ds_gemini:
    missing = [f for f in REQUIRED_FIELDS if f not in row]
    assert not missing, f"missing fields: {missing}"
    chunks_gemini.append({k: row[k] for k in REQUIRED_FIELDS})

print(f"Loaded {len(chunks_gemini)} chunks from {DEFAULT_CHUNKS_REPO}")
from collections import Counter
per_book = Counter(c["book_id"] for c in chunks_gemini)
print()
print("Per-book breakdown (Gemini OCR):")
for bid, n in sorted(per_book.items(), key=lambda kv: -kv[1]):
    print(f"  {bid:<22} {n:>4} chunks")
print()
print("First chunk preview:")
first = chunks_gemini[0]
print(f"  source_text     : {first['source_text']}")
print(f"  chapter / verse : {first['chapter']} / {first['verse_or_section']}")
print(f"  text (first 200): {first['text'][:200]!r}")


In [ ]:
# Phase 3b.2 -- Cell B: re-embed and upsert into a SEPARATE ChromaDB
# collection so the comparison run does NOT clobber any in-process
# `collection` variable created by Cell 6 above. Expect 206 vectors.
from src.rag.corpus_loader import build_chroma

collection_gemini = build_chroma(
    chunks_gemini,
    persist_dir="corpus/vector_db_gemini",
    collection_name="iks_corpus_gemini",
)
print()
print(f"ChromaDB ready: count={collection_gemini.count()} (expected 206)")
assert collection_gemini.count() == len(chunks_gemini), \
    f"Chroma count mismatch: {collection_gemini.count()} vs {len(chunks_gemini)}"


In [ ]:
# Phase 3b.2 -- Cell C: hybrid retriever (dense + sparse + reranker)
# over the new collection. Same configuration as cell 7 above so the
# comparison is apples-to-apples.
from src.rag.retriever import HybridRetriever

retriever_gemini = HybridRetriever(
    collection_gemini,
    use_dense=True, use_sparse=True, use_reranker=True,
)
print(
    f"retriever_gemini ready: dense={retriever_gemini.use_dense} "
    f"sparse={retriever_gemini.use_sparse} reranker={retriever_gemini.use_reranker}"
)


In [ ]:
# Phase 3b.2 -- Cell D: re-run the SAME 5 demo queries from cell 10
# above with the Gemini-OCR pipeline. Compare the answers, citations,
# and retrieved chunks against the Tesseract-era outputs in cell 10.
#
# Reuses the existing `generator` (Llama-3.1-8B-Instruct 4-bit) if it
# was already loaded earlier in this Colab session. If not, loads it
# fresh -- one-time ~5 min cost.
import torch

try:
    generator  # noqa: F821 -- re-use if cell 9 already ran
    print("Reusing already-loaded `generator` from earlier in the session.")
except NameError:
    print("`generator` not in memory -- loading Llama-3.1-8B 4-bit fresh ...")
    from src.rag.generator import GroundedGenerator
    generator = GroundedGenerator(
        model_name="meta-llama/Llama-3.1-8B-Instruct",
        load_in_4bit=True,
        temperature=0.2,
        max_new_tokens=512,
        seed=42,
    )
    generator._ensure_loaded()  # noqa: SLF001
    torch.cuda.empty_cache()
    mem = torch.cuda.memory_allocated() / 1024**3
    print(f"Llama loaded. CUDA memory in use: {mem:.2f} GiB")

from src.rag.pipeline import RAGPipeline
pipeline_gemini = RAGPipeline(
    retriever=retriever_gemini, generator=generator, default_k=5,
)

DEMO_QUERIES = [
    # Same five queries as cell 10:
    "How should a diseased tree with falling branches be treated?",
    "What signs in the sky predict imminent rainfall?",
    "How does the classical text guide finding underground water?",
    "What organic protocol should be used for a tree showing yellow leaves growing in mixed sandy-loam soil?",
    "What is the recommended drone-spraying schedule for monoculture rice fields?",
]

for q in DEMO_QUERIES:
    print("=" * 78)
    print(f"QUERY [GEMINI-OCR corpus]: {q}")
    result = pipeline_gemini.answer(q, k=5)
    print("--- ANSWER ---")
    print(result.answer)
    print("--- CITATIONS ---")
    for c in result.citations:
        print("  -", c)
    print("--- CHUNKS USED ---")
    for cid in result.used_chunk_ids:
        match = next((r for r in result.retrieved if r.chunk_id == cid), None)
        if match is not None:
            meta = match.metadata
            print(f"  {cid[:10]}  {meta.get('source_text')} ch.{meta.get('chapter')} v.{meta.get('verse_or_section')}")
    print("--- TOP RETRIEVED (for inspection) ---")
    for i, h in enumerate(result.retrieved[:3], 1):
        meta = h.metadata
        src = f"{meta.get('source_text','?')} ch.{meta.get('chapter','?')} v.{meta.get('verse_or_section','?')}"
        snip = (h.text or '').replace('\n',' ')[:120]
        print(f"  [{i}] score={h.score:.4f} {src} :: {snip}")
    print()


## How to compare the two runs

For each of the five queries in cell D above, look at the **same**
query in cell 10 (Tesseract-era) and contrast:

| What to compare | Tesseract era (cell 10) | Gemini era (cell D) |
|---|---|---|
| Did the model answer or refuse? | Q2 (rainfall) and Q3 (water) often refused or gave thin answers because chunks were Devanagari-noisy | Should now produce a real grounded answer for in-corpus questions |
| Citations | Heavy on Vrik + Brihat only (the only two books) | Can now also cite **Krishi Parashara** and **Upavanavinoda** |
| Retrieved chunk text | Look for stray Devanagari characters or garbled diacritics | Clean English, with proper diacritics (Asvattha, Prajapati, etc.) |
| Out-of-corpus Q5 (drones) | MUST emit the locked refusal sentence | Should still emit the same refusal -- this is the faithfulness guard |

If any **in-corpus** query that worked above now refuses, that is a
regression to flag. If any **out-of-corpus** query stops refusing and
starts hallucinating, that is a far more serious regression.

### Cost + provenance receipt for the thesis

- OCR engine    : Gemini 3.5 Flash (paid, gemini-3.5-flash)
- Pages OCR'd   : 466 across 4 books (Vrik 101 + Brihat 319 selected + KP 26 + UV 20)
- Spend         : ~Rs 41 INR (~$0.49 USD)
- Re-run cost   : free -- chunks pulled from the same private HF dataset
- Resume safety : per-page on-disk cache made the run idempotent across
                  3 sessions (one was killed by an IDE close, one by a
                  free-tier 429 before billing propagated)
- Pipeline      : unchanged from cell 5-9 above (hybrid retrieval + Llama-3.1-8B 4-bit grounded generator with the locked SYSTEM_PROMPT_V17)
